# K5 — Screening scenarios and full runs

This page checks phase K5: the screening programs of `data/scenarios.py` as summer4 flows,
and posterior full runs across scenarios written in the original's file formats.

**Claims to check.**

1. With the golden parameters, PEARL at 84.5% coverage (`scenario_3`) and CXR-TST at 84.5%
   (`scenario_8`, which sends TST positives to `cleared`) match the original's compartments and
   every derived output to 1e-5. The screening pulse in 2026 should be visible, and identical,
   in both models.
2. Full runs of posterior draws produce `uncertainty_df_<scenario>.parquet` and
   `diff_quantiles_df_ref_baseline_<scenario>.parquet` with the original's schema: averted
   burden is baseline minus scenario, so screening averts a positive number of cases.

In [ ]:
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.io as pio
import yaml

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from kiribati_tb.analysis import full_runs, saved_names
from kiribati_tb.calibration import calibration_setup
from kiribati_tb.model import ModelConfig, compile_model
from kiribati_tb.outputs import build_outputs, run_outputs
from kiribati_tb.paths import GOLDEN, REPO_ROOT
from kiribati_tb.scenarios import SCENARIOS, SCENARIOS_BY_ID

sys.path.insert(0, str(REPO_ROOT / "tests"))
from parity import parity_report, read_golden

pd.DataFrame(
    [
        {"id": s.sc_id, "name": s.sc_name, "programs": ", ".join(p.name for p in s.programs)}
        for s in SCENARIOS
    ]
)

## Scenario parity

Screening flows per year and TB incidence around the 2026 campaign, port solid, original
dashed, for both scenario fixtures.

In [ ]:
frames = {}
for fixture, sc_id in {"hetero_scenario_3": "scenario_3", "hetero_scenario_8": "scenario_8"}.items():
    params = yaml.safe_load((GOLDEN / fixture / "params.yaml").read_text())["params"]
    config = ModelConfig(screening=SCENARIOS_BY_ID[sc_id].programs)
    frames[fixture] = (
        run_outputs(compile_model(config), build_outputs(config), params),
        read_golden(fixture, "derived"),
    )

lines = {}
for fixture, (got, gold) in frames.items():
    for output in ("screening", "tb_incidence"):
        lines[f"{fixture}: {output} (summer4)"] = got.loc[2020:, output]
        lines[f"{fixture}: {output} (summer2gen)"] = gold.loc[2020:, output]
fig = pd.DataFrame(lines).plot(
    title="Screening and TB incidence around the 2026 campaign",
    labels={"index": "year", "value": "people per year", "variable": "series"},
)
for trace in fig.data:
    if "summer2gen" in trace.name:
        trace.line.dash = "dash"
fig

In [ ]:
reports = {f: parity_report(g, e, atol=1e-8) for f, (g, e) in frames.items()}
for fixture, report in reports.items():
    assert sorted(frames[fixture][0].columns) == sorted(frames[fixture][1].columns)
    assert report["passes"].all(), (fixture, report[~report["passes"]].head())
pd.DataFrame({f: r["max_rel"] for f, r in reports.items()}).plot.box(
    title="Max relative error per derived output, scenario fixtures",
    labels={"variable": "fixture", "value": "max relative error"}, log_y=True,
)

## Full runs in the original's formats

Eight draws around the golden parameters stand in for a posterior here (a real posterior comes
from `pixi run calibrate`). The bars are the median TB cases and deaths averted by 2035 in
each scenario relative to baseline; all should be positive, and PEARL (1–3) should avert more
than CXR only (16–18) at the same coverage.

In [ ]:
setup = calibration_setup()
params = yaml.safe_load((GOLDEN / "hetero_baseline" / "params.yaml").read_text())["params"]
rng = np.random.default_rng(1)
names = [p.name for p in setup.priors] + ["mixing_dist_sd"]
draws = {k: params.get(k, 12.5) * (1.0 + 0.02 * rng.standard_normal(8)) for k in names}
folder = Path(tempfile.mkdtemp())
scenario_ids = ["scenario_3", "scenario_8", "scenario_18"]
runs = full_runs(setup, draws, folder, scenario_ids, n=8, batch_size=8)
print(sorted(p.name for p in folder.iterdir()))

unc = pd.read_parquet(folder / "uncertainty_df_scenario_3.parquet")
assert unc.columns.names == ["output", "quantile"]
assert set(unc.columns.get_level_values(0)) == set(saved_names(setup.config))
averted = {
    sc: pd.read_parquet(folder / f"diff_quantiles_df_ref_baseline_{sc}.parquet").loc[0.5]
    for sc in scenario_ids
}
averted = pd.DataFrame(averted).T[["TB_averted", "deaths_averted"]]
assert (averted > 0).all().all()
assert averted.loc["scenario_3", "TB_averted"] > averted.loc["scenario_18", "TB_averted"]
averted.plot.bar(
    barmode="group", title="Median TB cases and deaths averted by 2035 vs baseline",
    labels={"index": "scenario", "value": "averted", "variable": "outcome"},
)